In [6]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
REPO_ROOT = next((candidate for candidate in candidates if (candidate / "cort_si").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repo root containing the cort_si package.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from cort_si import (
    adaptive_source_selection,
    construct_Y,
    construct_Sigma,
    construct_folds,
    construct_test_statistic,
    gen_data,
    solve_cort_model,
)

RESULTS_DIR = REPO_ROOT / "run_experiment" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
p = 300
nS = 100
num_info_aux = 3
num_uninfo_aux = 2
K = num_info_aux + num_uninfo_aux
lambda_sel = 0.5
lambda0 = 1
T = 5
nT_list = [50, 75, 100]

num_runs = 1000
base_seed = 1
alpha = 0.05
true_beta_scale = 1.0
rho = 0.0
sigma_noise = 1.0
source_shift_sd = 0.3
covariate_shift = False
save_interval = 1
verbose = True

print(f"num_runs = {num_runs}")
print(f"fixed alpha = {alpha}")
print(f"Null model true_beta scale = {true_beta_scale}")
print(f"K = {K}, p = {p}, nS = {nS}")
print(f"Testing nT values: {nT_list}")

num_runs = 1000
fixed alpha = 0.05
Null model true_beta scale = 1.0
K = 5, p = 300, nS = 100
Testing nT values: [50, 75, 100]


In [8]:
def get_save_path(p_val, nS_val, nT_val, K, lambda_sel, lambda0):
    return RESULTS_DIR / f"DS_tpr_p{p_val}_nS{nS_val}_nT{nT_val}_K{K}_lamsel{lambda_sel}_lam0{lambda0}.npz"


def load_progress(filepath):
    if filepath.exists():
        data = np.load(filepath)
        # Hỗ trợ cả file cũ (chỉ có null_pvalues) và file mới (có thêm j_values)
        if 'j_values' in data and 'null_pvalues' in data:
            return list(data['j_values']), list(data['null_pvalues'])
        elif 'null_pvalues' in data:
            return [], list(data['null_pvalues'])
    return [], []

def save_progress(filepath, j_values, null_pvalues, p_val, nS_val, nT_val, num_runs_val, alpha_val):
    np.savez(
        filepath,
        j_values=np.asarray(j_values, dtype=int),
        null_pvalues=np.asarray(null_pvalues, dtype=float),
        p=np.asarray([p_val], dtype=int),
        nS=np.asarray([nS_val], dtype=int),
        nT=np.asarray([nT_val], dtype=int),
        num_runs=np.asarray([num_runs_val], dtype=int),
        alpha=np.asarray([alpha_val], dtype=float),
    )


In [9]:
def generate_null_instance(nT_val, iteration_seed):
    return gen_data.generate_data(
        p=p,
        nS=nS,
        nT=nT_val,
        K=K,
        true_beta=true_beta_scale,
        rho=rho,
        sigma_noise=sigma_noise,
        source_shift_sd=source_shift_sd,
        covariate_shift=covariate_shift,
        seed=iteration_seed,
        num_info_aux=num_info_aux,
        num_uninfo_aux= num_uninfo_aux
    )


def split_indices_in_half(num_rows, rng):
    shuffled = rng.permutation(num_rows)
    selection_idx, inference_idx = [np.sort(block.astype(int)) for block in np.array_split(shuffled, 2)]
    if selection_idx.size == 0 or inference_idx.size == 0:
        return None, None
    return selection_idx, inference_idx


def split_dataset_block(X, Y, Sigma, rng):
    selection_idx, inference_idx = split_indices_in_half(X.shape[0], rng)
    if selection_idx is None:
        return None

    Sigma = np.asarray(Sigma)
    return {
        "X_selection": X[selection_idx],
        "Y_selection": np.asarray(Y)[selection_idx],
        "Sigma_selection": Sigma[np.ix_(selection_idx, selection_idx)],
        "X_inference": X[inference_idx],
        "Y_inference": np.asarray(Y)[inference_idx],
        "Sigma_inference": Sigma[np.ix_(inference_idx, inference_idx)],
    }


def split_instance_for_datasplit(XS_list, YS_list, SigmaS_list, X0, Y0, Sigma0, split_seed):
    rng = np.random.default_rng(split_seed)

    target_split = split_dataset_block(X0, Y0, Sigma0, rng)
    if target_split is None:
        return None

    return {
        "XS_selection_list": XS_list,
        "YS_selection_list": YS_list,
        "SigmaS_selection_list": SigmaS_list,
        "X0_selection": target_split["X_selection"],
        "Y0_selection": target_split["Y_selection"],
        "Sigma0_selection": target_split["Sigma_selection"],
        "XS_inference_list": XS_list,
        "YS_inference_list": YS_list,
        "SigmaS_inference_list": SigmaS_list,
        "X0_inference": target_split["X_inference"],
        "Y0_inference": target_split["Y_inference"],
        "Sigma0_inference": target_split["Sigma_inference"],
    }


def compute_random_selected_datasplit_pvalue(X0, Y0, XS_list, YS_list, SigmaS_list, Sigma0, lambdak_list, split_seed, feature_seed):
    split_data = split_instance_for_datasplit(XS_list, YS_list, SigmaS_list, X0, Y0, Sigma0, split_seed)
    if split_data is None:
        return None

    folds = construct_folds(split_data["X0_selection"].shape[0], T=T, shuffle=False)
    I_obs = adaptive_source_selection(
        split_data["X0_selection"],
        split_data["Y0_selection"],
        split_data["XS_selection_list"],
        split_data["YS_selection_list"],
        folds,
        lambda_sel,
        verbose=verbose,
    )
    _, beta0_hat, _, _ = solve_cort_model(
        split_data["X0_selection"],
        split_data["Y0_selection"],
        split_data["XS_selection_list"],
        split_data["YS_selection_list"],
        I_obs,
        lambda0,
        lambdak_list,
        verbose=verbose,
        label="Observed CoRT fit on selection half",
    )
    M_obs = [idx for idx, value in enumerate(beta0_hat) if value != 0]
    if not M_obs:
        return None

    feature_idx = random.Random(feature_seed).choice(M_obs)
    Y_inference = construct_Y(split_data["YS_inference_list"], split_data["Y0_inference"])
    Sigma_inference = construct_Sigma(split_data["SigmaS_inference_list"], split_data["Sigma0_inference"])
    X0M_inference = split_data["X0_inference"][:, M_obs]
    etaj, etajTy = construct_test_statistic(
        feature_idx,
        X0M_inference,
        Y_inference,
        M_obs,
        split_data["Y0_inference"].shape[0],
        Y_inference.shape[0],
    )

    variance = float(np.asarray(etaj.T @ Sigma_inference @ etaj).item())
    if not np.isfinite(variance) or variance <= 0.0:
        return None

    z_score = float(etajTy / np.sqrt(variance))
    datasplit_pvalue = float(2.0 * stats.norm.sf(abs(z_score)))

    return int(feature_idx), datasplit_pvalue,



def summarize_fpr_at_alpha(pvalues, alpha):
    if pvalues.size == 0:
        return np.array([], dtype=float), np.array([], dtype=float)

    rejection_indicators = (pvalues <= alpha).astype(float)
    running_fpr = np.cumsum(rejection_indicators) / np.arange(1, rejection_indicators.size + 1)
    return rejection_indicators, running_fpr


def run_resumable_tpr_experiment(nT_val, total_runs, seed_base, save_intvl=10):
    filepath = get_save_path(p, nS, nT_val, K, lambda_sel, lambda0)
    j_values, null_pvalues = load_progress(filepath)
    start_iteration = len(null_pvalues)

    print(f"Starting/Resuming experiment for nT={nT_val} from run {start_iteration}/{total_runs}...")

    cnt = 0
    sum = 0
    for iteration in range(start_iteration, total_runs):
        data_seed = base_seed + iteration
        XS_list, YS_list, X0, Y0, _, SigmaS_list, Sigma0, _ = generate_null_instance(nT_val, data_seed)
        lambdak_list = [lambda0] * len(XS_list)
        result = compute_random_selected_datasplit_pvalue(
            X0,
            Y0,
            XS_list,
            YS_list,
            SigmaS_list,
            Sigma0,
            lambdak_list,
            split_seed=data_seed + 10000,
            feature_seed=data_seed + 20000,
        )

        if result is not None:
            j_val, p_sel = result
            if p_sel is not None and np.isfinite(p_sel):
                if j_val <= 5:
                    if p_sel <= alpha:
                        cnt += 1
                    sum += 1
                j_values.append(int(j_val))
                null_pvalues.append(float(p_sel))

        if (iteration + 1) % save_intvl == 0 or (iteration + 1) == total_runs:
            save_progress(filepath, j_values, null_pvalues, p, nS, nT_val, total_runs, alpha)
            print(f"  [nT={nT_val}] Saved progress: {iteration + 1}/{total_runs} runs completed.")
            if sum > 0:
                print(f"Empirical TPR @ alpha={alpha}: {(cnt / sum):.3f} | total run : {sum}")

    return j_values, null_pvalues

In [10]:
import gc
for current_nT in nT_list:
    run_resumable_tpr_experiment(current_nT, num_runs, base_seed, save_intvl=save_interval)
    gc.collect()

Starting/Resuming experiment for nT=50 from run 29/1000...
len of active set: 3
len of active set: 1
source 0, fold 1: vote = True
len of active set: 8
len of active set: 1
source 0, fold 2: vote = True
len of active set: 7
len of active set: 1
source 0, fold 3: vote = False
len of active set: 8
len of active set: 1
source 0, fold 4: vote = False
len of active set: 6
len of active set: 1
source 0, fold 5: vote = False
source 0: votes = 2/5
len of active set: 3
len of active set: 0
source 1, fold 1: vote = True
len of active set: 8
len of active set: 0
source 1, fold 2: vote = False
len of active set: 7
len of active set: 1
source 1, fold 3: vote = False
len of active set: 8
len of active set: 0
source 1, fold 4: vote = False
len of active set: 6
len of active set: 0
source 1, fold 5: vote = False
source 1: votes = 1/5
len of active set: 3
len of active set: 0
source 2, fold 1: vote = True
len of active set: 8
len of active set: 0
source 2, fold 2: vote = False
len of active set: 7
len 

KeyboardInterrupt: 